# FaultSimulation(Ikeda)

## 2.Transition Fault

### import

In [1]:
import numpy as np
import os
from multiprocessing import Pool
from kyupy import verilog, cdiv
from kyupy.logic import unpackbits, packbits
from kyupy.logic_sim import LogicSim
from kyupy.techlib import NANGATE45

### 8値論理

In [2]:
def worker_sim_8valued_transition(task_info, data):
    line_idx, fault_type = task_info
    a_v1, b_v1, s_v1, a_v2, b_v2, s_v2, golden_v2 = data
    test_count = len(a_v1)
    
    # ----------------------------------------------------
    # ステップ1: Golden Run
    # ----------------------------------------------------
    fma.setup_input(a_v1, b_v1, s_v1, a_v2, b_v2, s_v2)
    fma.s_to_c()
    
    is_active = [False]
    
    # コールバック関数: 引数 line_id はオブジェクトではなく整数のインデックス
    def probe_cb(line_id, vals_bp):
        if line_id == line_idx:
            # vals_bp は [bit0(Final), bit1(Initial), bit2(Activity)] の3ビット構成
            if fault_type == 0:
                # RISE (0->1) : Activity=1, Initial=0, Final=1
                mask = vals_bp[2] & np.bitwise_not(vals_bp[1]) & vals_bp[0]
            else:
                # FALL (1->0) : Activity=1, Initial=1, Final=0
                mask = vals_bp[2] & vals_bp[1] & np.bitwise_not(vals_bp[0])
            
            # 1つでも遷移(Rise/Fall)が起きていれば活性化とみなす
            if np.any(mask):
                is_active[0] = True
                
    fma.c_prop(inject_cb=probe_cb)
    
    # 遷移が1つも起きなかった場合は検出不可として即リターン
    if not is_active[0]:
        return (fault_type, 0)
        
    # ----------------------------------------------------
    # ステップ2: Fault Injection
    # ----------------------------------------------------
    fma.s_to_c() # 状態をリセット
    
    def inject_cb(line_id, vals_bp):
        if line_id == line_idx:
            if fault_type == 0:
                # Slow-to-Rise: 実際にRISEした箇所だけを ZERO(Stuck-at-0) に引き戻す
                mask = vals_bp[2] & np.bitwise_not(vals_bp[1]) & vals_bp[0]
                vals_bp[2] &= np.bitwise_not(mask)
                vals_bp[0] &= np.bitwise_not(mask)
            else:
                # Slow-to-Fall: 実際にFALLした箇所だけを ONE(Stuck-at-1) に引き戻す
                mask = vals_bp[2] & vals_bp[1] & np.bitwise_not(vals_bp[0])
                vals_bp[2] &= np.bitwise_not(mask)
                vals_bp[0] |= mask

    fma.c_prop(inject_cb=inject_cb)
    fma.c_to_s()
    
    faulty_v2 = fma.get_output(test_count)
    
    # 遷移によって活性化し、かつ出力エラーとして観測されたか
    is_detected = np.any(faulty_v2 != golden_v2)
    
    return (fault_type, 1 if is_detected else 0)

### simulation

In [3]:
class FMA_Fixed(LogicSim):
    def __init__(self, netlist_file, sims=1024):
        assert sims > 0 and sims % 8 == 0
        circuit = verilog.load(netlist_file, branchforks=False, tlib=NANGATE45)
        circuit.resolve_tlib_cells(NANGATE45)
        
        # m=8 を指定し、KyuPyをネイティブな8値論理シミュレータとして起動
        super().__init__(circuit, sims, m=8, c_reuse=False, strip_forks=False)

        self.r_locs = self.circuit.io_locs("o_sum")
        self.bit_width = len(self.r_locs)
        
        if self.bit_width <= 8: self.data_format = np.int8
        elif self.bit_width <= 16: self.data_format = np.int16
        elif self.bit_width <= 32: self.data_format = np.int32
        else: self.data_format = np.int64
            
        self.a_locs = self.circuit.io_locs("i_weight")
        self.b_locs = self.circuit.io_locs("i_activation")
        self.s_locs = self.circuit.io_locs("i_sum")
        
        self.s[0] = 0

    def setup_input(self, a_v1, b_v1, s_v1, a_v2, b_v2, s_v2):
        nbytes = cdiv(len(a_v1), 8)
        
        def pack_input(locs, v1_data, v2_data):
            # V1とV2のデータをビット展開
            v1_bits = np.unpackbits(v1_data.view(np.uint8).reshape(-1, 4), axis=1)[:, ::-1][:, :len(locs)].T
            v2_bits = np.unpackbits(v2_data.view(np.uint8).reshape(-1, 4), axis=1)[:, ::-1][:, :len(locs)].T
            
            # ビットパッキング
            v1_packed = np.packbits(np.ascontiguousarray(v1_bits), axis=-1)
            v2_packed = np.packbits(np.ascontiguousarray(v2_bits), axis=-1)
            act_packed = v1_packed ^ v2_packed # V1とV2の排他的論理和でActivity(遷移)を計算
            
            # 8値論理の bp (bit-parallel) フォーマットに格納
            # idx 0: bit0 (Final), idx 1: bit1 (Initial), idx 2: bit2 (Activity)
            self.s[0, locs, 0, :nbytes] = v2_packed
            self.s[0, locs, 1, :nbytes] = v1_packed
            self.s[0, locs, 2, :nbytes] = act_packed

        pack_input(self.a_locs, a_v1, a_v2)
        pack_input(self.b_locs, b_v1, b_v2)
        pack_input(self.s_locs, s_v1, s_v2)

    def get_output(self, test_count):
        nbytes = cdiv(test_count, 8)
        # 出力結果から、最終状態である bit0 (idx=0) だけを抽出
        rb_final = np.unpackbits(self.s[1, self.r_locs, 0, :nbytes], axis=-1)
        
        target_bits = np.dtype(self.data_format).itemsize * 8
        rb_padded = np.zeros((target_bits, rb_final.shape[1]), dtype=np.uint8)
        rb_padded[:len(self.r_locs), :] = rb_final
        
        r_out = packbits(rb_padded.T, dtype=self.data_format).flatten()[:test_count]
        return r_out

### 実行部

In [4]:
if __name__ == "__main__":
    netlist_path = os.path.join('circuits', 'fma_fixed_8_24.gl.v')
    
    if not os.path.exists(netlist_path):
        print(f"Error: Netlist not found at {netlist_path}")
    else:
        global fma
        fma = FMA_Fixed(netlist_path, sims=1024)
        print(f"Circuit Loaded: {netlist_path}")

        test_count = 1024
        np.random.seed(42)
        
        a_v1 = np.random.randint(-2147483648, 2147483647, size=test_count, dtype=np.int32)
        b_v1 = np.random.randint(-2147483648, 2147483647, size=test_count, dtype=np.int32)
        s_v1 = np.random.randint(-2147483648, 2147483647, size=test_count, dtype=np.int32)
        
        a_v2 = np.random.randint(-2147483648, 2147483647, size=test_count, dtype=np.int32)
        b_v2 = np.random.randint(-2147483648, 2147483647, size=test_count, dtype=np.int32)
        s_v2 = np.random.randint(-2147483648, 2147483647, size=test_count, dtype=np.int32)
        
        print("Running Golden Simulation (Native 8-valued V1->V2)...")
        fma.setup_input(a_v1, b_v1, s_v1, a_v2, b_v2, s_v2)
        fma.s_to_c()
        fma.c_prop()
        fma.c_to_s()
        golden_v2 = fma.get_output(test_count)

        sample_size = 100
        all_lines = list(range(len(fma.circuit.lines)))
        targets = np.random.choice(all_lines, min(sample_size, len(all_lines)), replace=False)

        print(f"\nExperiment: Native 8-Valued Transition Fault Simulation")
        print(f"  Target Lines: {sample_size} (STR and STF)")
        print("-" * 50)

        data_packet = (a_v1, b_v1, s_v1, a_v2, b_v2, s_v2, golden_v2)
        
        tasks = []
        for l in targets:
            tasks.append((l, 0)) # STR
            tasks.append((l, 1)) # STF

        with Pool(processes=8) as pool:
            results = pool.starmap(worker_sim_8valued_transition, [(t, data_packet) for t in tasks])

        det_str = sum(1 for r in results if r[0] == 0 and r[1] > 0)
        det_stf = sum(1 for r in results if r[0] == 1 and r[1] > 0)
        
        rate_str = det_str / sample_size * 100
        rate_stf = det_stf / sample_size * 100
        total_rate = (det_str + det_stf) / (2 * sample_size) * 100
        
        print(f"Slow-to-Rise (STR) Detection: {rate_str:.1f}%")
        print(f"Slow-to-Fall (STF) Detection: {rate_stf:.1f}%")
        print(f"Total Transition Fault Coverage: {total_rate:.1f}%")
        print("-" * 50)

Circuit Loaded: circuits/fma_fixed_8_24.gl.v
Running Golden Simulation (Native 8-valued V1->V2)...

Experiment: Native 8-Valued Transition Fault Simulation
  Target Lines: 100 (STR and STF)
--------------------------------------------------
Slow-to-Rise (STR) Detection: 98.0%
Slow-to-Fall (STF) Detection: 99.0%
Total Transition Fault Coverage: 98.5%
--------------------------------------------------
